# From First Eval to Autonomous AI Ops

This notebook is the hands-on companion to *"From First Eval to Autonomous AI Ops: A Maturity Model for AI Evaluation."* It walks you through the four stages of building an evaluation practice -- from cold-start instrumentation to AI-assisted eval ops. By the end, you'll have traced an LLM app, identified failure modes, and built an evaluator using either the Arize platform UI with the Alyx copilot or the AX CLI with an AI coding agent.

The app you'll instrument is a company-name entity extractor: it takes search queries, detects company names, and maps them to stock ticker symbols using OpenAI. Every LLM call is traced to **Arize AX** via OpenInference auto-instrumentation.

## Create your Arize account

1. Go to [app.arize.com/auth/join](https://app.arize.com/auth/join) and create a free account.
2. Once signed in, go to **Settings → Space API Keys** to copy your **Space ID** and **API Key**.

## Before you start

You'll need three keys ready to paste:

| What | Where to get it |
|------|----------------|
| **OpenAI API Key** | [platform.openai.com/api-keys](https://platform.openai.com/api-keys) |
| **Arize Space ID** | Settings → Space API Keys (after signing up above) |
| **Arize API Key** | Same page as Space ID |

In [ ]:
!pip install -q openai==2.30.0 arize-otel==0.11.0 openinference-instrumentation==0.1.46 openinference-instrumentation-openai==0.1.43

In [ ]:
import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
os.environ["ARIZE_SPACE_ID"] = getpass.getpass("Arize Space ID: ")
os.environ["ARIZE_API_KEY"] = getpass.getpass("Arize API Key: ")

print("Keys set.")

In [ ]:
from arize.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry import trace as otel_trace

PROJECT_NAME = "entity-transformer"

tracer_provider = register(
    space_id=os.environ["ARIZE_SPACE_ID"],
    api_key=os.environ["ARIZE_API_KEY"],
    project_name=PROJECT_NAME,
)
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

# Health check trace
with otel_trace.get_tracer(__name__).start_as_current_span("tracing_health_check"):
    print(f"Tracing enabled project '{PROJECT_NAME}'")

---

> **Stage 1 -- Crawl: Instrument and Observe**
>
> You're starting from zero. The first move is to get traces flowing so you can see what your LLM app is actually doing.

## Define the Pipeline

The `CompanyTickerTransformer` sends each query to `gpt-4o-mini` with a flat prompt (system instructions + query in one message) and parses the JSON response into a company-to-ticker mapping.

In [ ]:
import json
from dataclasses import dataclass, field

import openai
from openinference.instrumentation import using_prompt_template


@dataclass
class TransformResult:
    original_query: str
    entities: dict[str, str] = field(default_factory=dict)


COMPANY_TICKER_SYSTEM = """\
You are a financial entity extractor. Given a search query, identify all company \
names and return a JSON object mapping each company name (exactly as it appears \
in the query) to its stock ticker symbol.

Rules:
- Return ONLY valid JSON, no markdown fences, no preamble, no explanation.
- If no company names are found, return an empty object: {{}}.
- Use the exact string from the query as the key.
- Use the primary exchange ticker as the value (e.g. TSLA, AAPL, GOOGL).
- Do not include companies that are already written as tickers."""

PROMPT_TEMPLATE = """\
{system}

Query: {query}"""


class CompanyTickerTransformer:
    """Extracts company names and maps them to ticker symbols using a flat prompt."""

    entity_type = "company_ticker"

    def __init__(self, client: openai.OpenAI) -> None:
        self._client = client

    def detect_entities(self, query: str) -> TransformResult:
        prompt = PROMPT_TEMPLATE.format(
            system=COMPANY_TICKER_SYSTEM.replace("{{}}", "{}"), query=query
        )
        with using_prompt_template(
            template=PROMPT_TEMPLATE,
            variables={"system": COMPANY_TICKER_SYSTEM, "query": query},
            version="v1",
        ):
            response = self._client.chat.completions.create(
                model="o3-mini-2025-01-31",
                messages=[{"role": "user", "content": prompt}],
            )

        raw = response.choices[0].message.content.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[-1] if "\n" in raw else raw[3:]
            raw = raw.rsplit("```", 1)[0].strip()

        try:
            mapping: dict[str, str] = json.loads(raw)
        except json.JSONDecodeError:
            print(f"  [warn] could not parse JSON: {raw!r}")
            mapping = {}

        entities = {
            company: ticker.upper()
            for company, ticker in mapping.items()
            if company in query
        }
        return TransformResult(original_query=query, entities=entities)


class TransformPipeline:
    def __init__(self, transformers):
        self._transformers = transformers

    def run(self, query: str) -> TransformResult:
        all_entities: dict[str, str] = {}
        for transformer in self._transformers:
            result = transformer.detect_entities(query)
            all_entities.update(result.entities)
        return TransformResult(original_query=query, entities=all_entities)


oai_client = openai.OpenAI()
pipeline = TransformPipeline([CompanyTickerTransformer(oai_client)])

print("Pipeline ready.")

## Run the Queries

*Still Stage 1 -- this is the "observe" half of Crawl. You're sending production-shaped traffic through the pipeline and collecting traces for every call.*

25 queries covering single companies, multiple companies, no companies, ambiguous names, already-ticker text, and edge cases. Each one gets traced to Arize.

In [ ]:
DEMO_QUERIES = [
    # Single company
    "Tesla announced new battery technology this week",
    "Apple reported record quarterly earnings",
    "How is Microsoft performing in cloud services?",
    # Multiple companies
    "Tesla and Apple are competing in the EV market",
    "Google and Meta are both investing heavily in AI",
    "Amazon, Netflix, and Disney are battling for streaming dominance",
    # No companies
    "What is the weather like in San Francisco today?",
    "Best hiking trails near Denver",
    "How do I make sourdough bread at home?",
    # Ambiguous names
    "Delta flights were delayed due to weather",
    "I love Apple pie especially in autumn",
    "How is United performing on cross-country routes?",
    "Can you recommend a good Shell gas station near me?",
    "I need to buy new Coach bags for the holiday season",
    "The Gap between rich and poor keeps growing every year",
    "My kids love going to Target practice after school",
    "We took a Carnival cruise through the Caribbean last summer",
    "I heard Sprint speeds have improved in the new stadium",
    "Is there a Subway sandwich shop open near Times Square?",
    # Already-ticker text
    "TSLA reported earnings, should I buy more?",
    "AAPL and MSFT both hit all-time highs today",
    # Multi-sentence
    "Google acquired a promising AI startup last week. Microsoft responded by announcing its own deal.",
    "Nvidia's chips power most AI data centers. AMD is trying to close the gap with new hardware.",
    # Edge cases
    "Berkshire Hathaway and JPMorgan Chase released their annual reports",
    "The Coca-Cola Company increased its dividend for the 60th consecutive year",
]

tracer = otel_trace.get_tracer(__name__)
total = len(DEMO_QUERIES)
print(f"Running {total} queries...\n")

for i, query in enumerate(DEMO_QUERIES, 1):
    with tracer.start_as_current_span("transform_query") as span:
        span.set_attribute("openinference.span.kind", "CHAIN")
        span.set_attribute("input.value", query)
        try:
            result = pipeline.run(query)
            output = json.dumps(result.entities, ensure_ascii=False)
            span.set_attribute("output.value", output)
            span.set_status(otel_trace.Status(otel_trace.StatusCode.OK))
        except Exception as e:
            span.set_status(otel_trace.Status(otel_trace.StatusCode.ERROR, str(e)))
            output = f"ERROR: {e}"
    print(f"[{i:02d}/{total}] {query}")
    print(f"         \u2192 {output}\n")

tracer_provider.force_flush()
print("\u2705 Done! Traces exported to Arize.")

---

> **Transition -- You can see the problems. Now what?**
>
> Crawl got you observability. You have traces, you can inspect inputs and outputs, and you can eyeball failures one at a time. But eyeballing doesn't scale. The next stages introduce AI-assisted evaluation to do this systematically.

## View Your Traces

Head to [app.arize.com](https://app.arize.com) and open the **entity-transformer** project.

Each query produced a two-span trace:

| Span | Kind |
|------|------|
| **transform_query** | CHAIN |
| &nbsp;&nbsp;&nbsp;&nbsp;**chat.completions.create** | LLM (auto-instrumented) |

Look at the outputs. Did the model get any wrong? Check the **ambiguous queries**:

| Query | What might go wrong |
|-------|--------------------|
| *"Delta flights were delayed due to weather"* | Delta Airlines mapped to DAL even though the query is about flights, not stocks |
| *"I love Apple pie especially in autumn"* | Apple mapped to AAPL even though this is about food |
| *"How is United performing on cross-country routes?"* | United Airlines mapped to UAL even though context is ambiguous |

These failures are intentional -- the flat prompt packs everything into a single user message,
making it harder for the model to distinguish instructions from the query. **Your next step
is to find these problems and fix them.**

## Choose Your Path

Now it's time to **analyze failure modes and build an evaluator**. Pick how you want to apply AI to this problem:

- **Path B** (Stage 2 -- Walk) — Use Alyx, the AI copilot built into the Arize platform, to do the same work directly in the UI. This is AI-assisted eval ops: you're collaborating with an AI copilot to analyze traces and build evaluators without writing code.
- **Path A** (Stage 3 -- Run) — Use an AI coding agent (Cursor, Claude Code, etc.) with the AX CLI to drive everything from your terminal. This is the headless developer workflow: your agent reads span data, designs the evaluator, and executes the full pipeline programmatically.

Both paths end in the same place -- an evaluator scoring your spans. Try one or both.

<details open>
<summary><h3>Path A: AX CLI + AI Coding Agent</h3></summary>

> **Stage 3 -- Run: Headless Developer Workflow**
>
> Your AI coding agent drives the entire evaluation pipeline from the terminal -- exporting spans, analyzing failures, building the evaluator, and triggering a backfill -- with no UI clicks required.

*For those comfortable with the command line and AI coding agents (Cursor, Claude Code, etc.).*

The AX CLI is Arize's command-line tool for working with projects, spans, evaluators, and more. It's also what powers the **Arize Skills** for AI coding agents — when you ask an agent to export spans or create an evaluator, it's running `ax` commands under the hood.

---

#### Set up the CLI

**Install the AX CLI** in your local terminal (not the notebook — your agent needs it on PATH):

```bash
uv tool install arize-ax-cli
```

Or if you don't have `uv`: `pip install arize-ax-cli`

**Create a working directory** to keep exported data tidy:

```bash
mkdir -p .scratch
```

**Set up your profile and environment.** Save your API key so the CLI can authenticate, and export your Space ID:

```bash
ax profiles create --api-key YOUR_ARIZE_API_KEY
export ARIZE_SPACE_ID="YOUR_SPACE_ID"
```

Use the same API Key and Space ID you entered in the notebook setup above.

**Explore the CLI.** Type `ax` and press Tab to see all commands (tab completion is built in). Or run:

```bash
ax --help
```

**Find your project.** List the projects in your space and find the **entity-transformer** row. Copy its **ID** (a base64 string like `TW9kZWw6...`) — you'll need it for the export command. The human-readable name won't work as an argument.

```bash
ax projects list
```

**Export your spans** into your working directory, using the **project ID** from the list above:

```bash
ax spans export YOUR_PROJECT_ID --output-dir .scratch
```

---

#### Use your AI agent

**Install Arize Skills** in your agent so it knows how to use the AX CLI:

```bash
npx skills add Arize-ai/arize-skills --skill '*' --yes
```

This requires Node.js/npm. If you don't have it, your agent can install the skills itself when you give it a task — just mention "use Arize skills."

**Step 1: Set up an OpenAI AI integration.** Evaluators need LLM credentials to run the judge model. Ask your agent:

> *"Set up an OpenAI AI integration in my Arize space so I can use gpt-4o as a judge model for evaluations."*

The agent will use `ax ai-integrations create` with your OpenAI API key. This stores the credentials in Arize so evaluators can call the judge model on your behalf.

---

**Step 2: Analyze failure modes.** Paste this prompt into your AI agent:

> *"Look at the exported spans in `.scratch/`. Analyze the entity-transformer traces and tell me about the failure modes — which queries produced incorrect entity extractions, and why?"*

The agent will read the exported JSON, identify the ambiguous-name failures (Delta, Apple pie, United), and summarize the patterns.

---

**Step 3: Plan the evaluator.** Before letting your agent execute anything, switch to **plan mode** (in Cursor: click the mode selector and choose "Plan"). This forces the agent to study the exported span data and design a correct approach *before* making API calls.

Give your agent this goal-oriented prompt and let it figure out the details:

> *"I want to create an LLM-as-judge evaluator called 'Entity Extraction Correctness' and then run it as a one-time backfill against my entity-transformer project. The evaluator should judge whether the extracted entities are company names that were actually mentioned in the original query. Use my OpenAI integration and gpt-4o as the judge model.*
>
> *Look at the exported spans in `.scratch/` to understand the span schema — what attributes are available on the LLM spans, what the input and output look like, and how the query filter and column mappings should be configured.*
>
> *Make a plan covering: the evaluator template, the task query filter, and the column mappings. Don't execute yet."*

Review the plan the agent produces. It should address:
- What span attributes to use for the evaluator's `{input}` and `{output}` template variables
- How to filter the task to only score LLM spans (not chain spans)
- The judge prompt template and classification labels

If the plan looks right, move on to Step 4. If not, ask the agent to revise.

---

**Step 4: Execute the plan.** Switch back to **agent mode** and tell it to go:

> *"Execute the plan. Create the evaluator, create the evaluation task, and trigger a backfill run over all historical spans. Wait for the run to complete."*

The agent will create the evaluator via `ax evaluators create`, wire up a task with column mappings and a query filter via `ax tasks create`, then trigger and monitor a backfill run.

**If the run fails or skips all spans**, check these common pitfalls:

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| All spans skipped, 0 errors | Column mapping paths don't resolve to actual span attributes | Have the agent re-read the exported spans and correct the paths |
| Filter matches 0 spans | Wrong query filter syntax (e.g. `span_kind = 'LLM'`) | The filter dimension must use the full attribute path: `attributes.openinference.span.kind = 'LLM'` |
| Run cancelled after ~1 second | AI integration credentials invalid | Check the integration via `ax ai-integrations list` |
| Run cancelled after ~3 minutes | Judge model call failed | Verify the model name (e.g. `gpt-4o`) and integration key |

If you hit an issue, paste the error back to your agent and ask it to fix it — that's the real-world workflow.

---

**Step 5: Check results.** Head to [app.arize.com](https://app.arize.com), open **entity-transformer**, and look at the evaluation scores on your LLM spans. Filter by the `incorrect` label to see exactly which queries the model got wrong.

</details>

<details>
<summary><h3>Path B: Arize Platform + Alyx Copilot</h3></summary>

> **Stage 2 -- Walk: AI-Assisted Eval Ops**
>
> You're collaborating with an AI copilot inside the platform. Alyx analyzes your traces, identifies failure patterns, and helps you build an evaluator -- all without leaving the browser.

*For everyone -- use the Arize UI with AI assistance built in.*

---

**Step 1: Open your project.** Navigate to [app.arize.com](https://app.arize.com) and open the **entity-transformer** project. You'll see the list of traces from the queries you just ran — each one shows a `transform_query` chain span with a child LLM span.

Feel free to click into any trace and explore the input/output — but the real power is using Alyx to do this at scale.

---

**Step 2: Ask Alyx to analyze your traces.** Open the Alyx copilot (bottom-right of the page) and ask:

> *"Analyze these traces. What's going on? Identify the failure modes — which queries have incorrect entity extractions and why?"*

Alyx will scan your spans, identify the ambiguous-name failures (Delta as airline vs. stock, Apple pie vs. AAPL, United flights vs. UAL), and give you a summary with suggestions for improvement.

---

**Step 3: Set up an OpenAI AI integration.** Evaluators need LLM credentials to call the judge model. Go to **Settings → AI Integrations** and click **New Integration**:

- **Provider:** OpenAI
- **Name:** e.g. "OpenAI"
- **API Key:** Paste your OpenAI API key

This stores the credentials in Arize so evaluators can call gpt-4o (or another model) on your behalf.

---

**Step 4: Create an evaluator.** Right from the same project page, you can create an evaluator. Click **Evaluators** in the left nav, or ask Alyx:

> *"Help me create an evaluator for entity extraction correctness."*

Walk through the creation flow:

- **Name:** Entity Extraction Correctness
- **Template:** A judge prompt that checks whether the extracted entities are actually company names in the original query (Alyx can draft this for you)
- **Labels:** `correct` / `incorrect`
- **Model:** Pick your LLM judge (e.g. gpt-4o via your OpenAI integration)

<!-- Screenshot placeholder: evaluator creation UI -->

---

**Step 5: Run the evaluator on your data.** Once the evaluator is created, run it against your LLM spans as a backfill. Alyx can walk you through this, or use the **Tasks** UI to create a one-time evaluation task targeting the `entity-transformer` project.

---

**Step 6: Review results.** Head back to your project's traces view. Each LLM span now has an evaluation score attached. Sort or filter by the `incorrect` label to see exactly which queries the model got wrong.

</details>

## You're Here

Look at how far you've come through the maturity model:

1. **Crawl** -- You instrumented an LLM app with OpenInference tracing, sent production-shaped traffic through it, and got traces flowing into Arize. You could see every input, output, and failure -- but only by eyeballing spans one at a time.
2. **Walk** -- You used Alyx (or the trace viewer) to analyze failure patterns at scale and built an LLM-as-judge evaluator that scores entity extraction correctness automatically.
3. **Run** -- You drove the same workflow headlessly from a CLI and AI coding agent: exporting spans, designing the evaluator, wiring up a task, and triggering a backfill -- all without touching the UI.

From here, the natural next step is to **improve the prompt** based on what the evaluator found -- separate the system instructions from the user query, add disambiguation rules, or try a different model. Re-run the queries, re-run the evaluator, and compare. But there's one more stage.

---

> **Stage 4 -- Fly: Autonomous AI Ops**

Everything you did in this notebook was manual: you triggered the queries, you (or your agent) kicked off the evaluator, you reviewed the results. Stage 4 removes the human from the loop.

Picture this: a **monitor** watches the evaluation score on your entity-transformer project. When the correctness rate drops below a threshold -- say, after a model update or a prompt change -- it fires a **webhook**. That webhook triggers an always-on agent that triages the regression: it pulls the failing spans, diagnoses the pattern, proposes a fix, and opens a PR or flags a human only when it can't resolve the issue autonomously.

You don't build that in a notebook. You build it with the same primitives you just learned -- traces, evaluators, tasks, and the AX CLI -- wired into a continuous pipeline. That architecture is the subject of the next post in this series.